# Quick Demo — 5 分鐘 RAG 入門

這個 notebook 是 [My-AI-Learning-Notes](https://github.com/markl-a/My-AI-Learning-Notes) 的「press play」入口：把整個 repo 教的概念，濃縮成最小可執行的範例。

## 為什麼存在？

這個 repo 90% 是繁中教材（markdown 文件、學習路徑、面試準備）——但訪客打開 GitHub 通常想 **5 分鐘看到一個能跑的東西**，再回頭讀文件。這個 notebook 就是那個 5 分鐘示範。

## 你會看到什麼

1. **LLM 基本呼叫**（對應 `1.從AI到LLM基礎/`）— 一行 prompt → 回應
2. **RAG retrieval** 簡單實作（對應 `3.LLM應用工程/`）— 從幾段文件抽出最相關的，餵給 LLM
3. **多輪對話**（對應 `2.深入LLM模型工程與LLM運維/`）— 維護 message 歷史

## 跑起來

- **沒有 API key 也能看**：GitHub 直接 render，每個 cell 都附了預期輸出。
- **本機跑**：`pip install anthropic && export ANTHROPIC_API_KEY=...` 然後 `jupyter notebook examples/quick_demo.ipynb`。
- **不會用 jupyter**：把每個 code cell 直接複製到 `python3 -c '...'` 也能跑。

下面五個 cell 5 分鐘讀完，回頭挑一個資料夾深入學。

---
## 1️⃣ 最小 LLM 呼叫

對應筆記：`1.從AI到LLM基礎/` 第一個概念——LLM 就是「文字進、文字出」。

In [ ]:
import os

def ask_llm(prompt: str) -> str:
    """最小可呼叫範例。沒有 ANTHROPIC_API_KEY 時回傳 mock 回應。"""
    if not os.environ.get("ANTHROPIC_API_KEY"):
        return f"[mock] 你問了：{prompt[:50]}…（設 ANTHROPIC_API_KEY 看真實回應）"
    from anthropic import Anthropic
    client = Anthropic()
    msg = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}],
    )
    return msg.content[0].text

print(ask_llm("用三句話解釋什麼是大型語言模型"))

---
## 2️⃣ 簡單 RAG（不用裝 vector db）

對應筆記：`3.LLM應用工程/` 的 RAG 一節。實際生產會用 FAISS / Chroma / pgvector，但**核心概念是「從一堆文件挑最相關的給 LLM 看」**——下面 30 行純 Python 就能示範。

In [ ]:
import re, hashlib

# 假裝這是你的知識庫
knowledge = [
    "Transformer 是 2017 年 Google 在「Attention is All You Need」提出的架構。",
    "GPT 系列由 OpenAI 訓練，使用的是 Transformer 的 decoder-only 版本。",
    "LLaMA 是 Meta 開源的模型家族，2023 年 LLaMA 1、2024 年 LLaMA 3 釋出。",
    "Chinchilla scaling law 主張同樣 compute 下，更小模型訓更多 tokens 比較划算。",
    "RAG 的核心是「檢索增強」——先從文件挑相關內容，再給 LLM 生成答案。",
]

def cheap_embed(text, dim=32):
    """非常 naive 的 embedding：每個 token 雜湊到一個 bucket。production 別這樣。"""
    vec = [0.0] * dim
    for tok in re.findall(r"\w+", text):
        h = int(hashlib.sha256(tok.encode()).hexdigest(), 16)
        vec[h % dim] += 1.0
    norm = sum(v*v for v in vec) ** 0.5 or 1.0
    return [v/norm for v in vec]

def cosine(a, b): return sum(x*y for x, y in zip(a, b))

def retrieve(question, docs, k=2):
    qv = cheap_embed(question)
    scored = [(cosine(qv, cheap_embed(d)), d) for d in docs]
    return sorted(scored, key=lambda x: -x[0])[:k]

question = "RAG 是什麼？"
matches = retrieve(question, knowledge)
for score, doc in matches:
    print(f"  [sim={score:.3f}] {doc}")

context = "\n".join(d for _, d in matches)
answer = ask_llm(f"基於以下知識回答問題。知識：\n{context}\n\n問題：{question}")
print("\n答案：", answer)

---
## 3️⃣ 多輪對話（保留 message 歷史）

對應筆記：`2.深入LLM模型工程與LLM運維/` 的 chat history 一節。stateless 的 LLM 怎麼「記得」前一輪？答案：每一輪把 history 全部當 context 重送一次。

In [ ]:
history = []

def chat_turn(user_text):
    history.append({"role": "user", "content": user_text})
    if not os.environ.get("ANTHROPIC_API_KEY"):
        reply = f"[mock] 我會記住你說了：{user_text[:30]}…"
    else:
        from anthropic import Anthropic
        client = Anthropic()
        msg = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=200,
            messages=history,
        )
        reply = msg.content[0].text
    history.append({"role": "assistant", "content": reply})
    return reply

print("User: 我叫小明，今年 25 歲。")
print("Bot :", chat_turn("我叫小明，今年 25 歲。"))
print("\nUser: 我幾歲？")
print("Bot :", chat_turn("我幾歲？"))
print("\n(history 長度:", len(history), "則訊息)")

---

## 接下來該讀哪邊？

看你想深入哪個方向：

| 想學的東西 | 進入點 |
|---|---|
| LLM 是怎麼訓的 / 評估 / 部署 | [`2.深入LLM模型工程與LLM運維/`](../2.深入LLM模型工程與LLM運維/) |
| 用 LLM 做產品（RAG / Agent / Tool）| [`3.LLM應用工程/`](../3.LLM應用工程/) |
| 2024-2025 年最新研究進展 | [`5.AI研究前沿_2024-2025/`](../5.AI研究前沿_2024-2025/) |
| 中文社群在討論什麼 | [`4.相關的更新Blog/`](../4.相關的更新Blog/) |

## 周邊專案

- 把這些觀念跑起來：[phantom-mesh](https://github.com/markl-a/phantom-mesh) — Rust 寫的多平台 agent runtime
- 用在資料分析上：[Data-Analysis-with-Agents](https://github.com/markl-a/Data-Analysis-with-Agents)
- 用在自動化上：[Automation_with_Agent](https://github.com/markl-a/Automation_with_Agent)